# 05 Regime and shock analysis (A2)

**Owner:** Member 4

**Checklist**

- [x] Read docs/regimes.md (committed before results)
- [x] Error by regime for every model
- [x] DM tests inside each regime
- [x] One figure per test window

Rules: logic lives in `src/`; clear outputs before committing; save figures with `src.viz.style.save`; numbers for the report go to `reports/tables/`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run everything from the repo root
sys.path.insert(0, ".")
import numpy as np, pandas as pd
from IPython.display import display
from src.viz.style import apply_style, save
from src.eval.walkforward import load_all_predictions
from src.eda import regime_analysis as ra
apply_style()
preds = load_all_predictions()          # run `python run.py models` first
assert len(preds), "no predictions yet: run `python run.py models`"
p = ra.attach_regimes(preds)            # regime of the day each forecast is made (known at that time)
tab = ra.error_table(p)
tab.to_csv("reports/tables/regime_errors.csv", index=False)
print(len(tab), "rows;", "models:", sorted(tab.model.unique()))

## 1. The pre-registered regimes (docs/regimes.md)
Volatility regimes use the 22-day volatility of the market proxy, with cut-offs fixed on the training years 2002 to 2017. The post-gap regime is the test2 window; the crisis window is 2020-03 to 2020-12 inside test1. These were committed before any model result existed.

In [ ]:
display(pd.read_csv("reports/tables/regime_summary.csv"))

**What we see.** The COVID window (test1) is mostly turbulent and has very few calm days, so its calm column rests on a handful of days and is weak evidence. The post-gap window (test2) is mostly normal, with a good number of calm and turbulent days.

## 2. Accuracy by regime, one figure per test window
The bars show out-of-sample R2 against the zero forecast: above 0 means the model beats predicting no change. Value labels mark bars cut off at the bottom of the axis.

In [ ]:
fig = ra.fig_regime_r2(tab, "test1")
save(fig, "regime_r2_test1")

**What we see (COVID window).** The machine-learning models (orange) and the simple AR model (blue) beat the zero forecast in the normal and turbulent regimes. The calm bars are based on only a few days, so the naive persistence forecast looking best there should not be trusted. The persistence forecast is far worse than zero on turbulent days.

**What it means.** Even in the COVID shock the models kept a small edge, but the edge is modest and at 5 days it nearly disappears for the random forest on turbulent days.

In [ ]:
fig = ra.fig_regime_r2(tab, "test2")
save(fig, "regime_r2_test2")

**What we see (post-gap window).** The machine-learning models beat the zero forecast in every regime and by the most on calm days. Their advantage shrinks as volatility rises. The AR model follows the same pattern with smaller gains. Persistence is much worse than zero except on calm days at 1 day ahead.

**What it means.** Predictability is highest when the market is quiet and lowest when it is turbulent, which fits the idea that large shocks carry little forecastable information.

## 3. The numbers behind the figures (market proxy)

In [ ]:
m = tab[(tab.entity_set == "MARKET") & (tab.regime_type == "volatility")]
for h in (1, 5):
    print(f"Out-of-sample R2 vs zero forecast, horizon {h}")
    display(m[m.horizon == h].pivot_table(index=["window", "model"], columns="regime", values="oos_r2")[ra.VOL_ORDER].round(3))
print("Direction accuracy, horizon 1 (share of days with the sign right)")
display(m[m.horizon == 1].pivot_table(index=["window", "model"], columns="regime", values="accuracy")[ra.VOL_ORDER].round(3))
print("Number of test days per regime")
display(m[(m.horizon == 1) & (m.model == "zero")].pivot_table(index="window", columns="regime", values="n_dates")[ra.VOL_ORDER])

## 4. The 2020 crisis window against the rest of the COVID test window

In [ ]:
c = tab[(tab.regime_type == "crisis") & (tab.entity_set == "MARKET")]
display(c.pivot_table(index=["horizon", "model"], columns="regime", values="oos_r2").round(3))
display(c[(c.horizon == 1) & (c.model == "zero")][["regime", "n_dates"]])

**What we see.** Inside the crisis months the machine-learning models still beat the zero forecast, with a slightly smaller gain than in the rest of the window. The AR model's gain is about the same in both.

**What it means.** The models did not break down during the 2020 shock; the effect is weaker, not reversed.

## 5. Diebold-Mariano tests inside each regime
Each model is compared with the best baseline inside the same regime, on squared errors averaged per day. A negative statistic means the model has the smaller error. The p-values are NOT corrected for the many tests run.

In [ ]:
v = tab[tab.regime_type == "volatility"].dropna(subset=["dm_p"])
sig = v[v.dm_p < 0.05]
better = sig[sig.dm_stat < 0]
print(f"{len(v)} comparisons; {len(sig)} significant at 5 percent; {len(better)} of those favour the model")
print("expected by chance alone at 5 percent:", round(0.05 * len(v), 1))
display(sig.groupby(["model", np.where(sig.dm_stat < 0, "model better", "model worse")]).size().unstack(fill_value=0))
display(better[["window", "entity_set", "horizon", "regime", "model", "best_baseline", "oos_r2", "dm_stat", "dm_p", "n_dates"]]
        .sort_values(["window", "entity_set", "horizon"]).reset_index(drop=True))

**What we see.** Where a model is significantly different from the best baseline, it is either a machine-learning or AR model doing better, or the naive persistence forecast (and occasionally the zero forecast) doing worse. No machine-learning or AR model is significantly worse than the best baseline in any regime.

**What it means.** The evidence for a real but small advantage is consistent across regimes, but individual p-values should be read with care: many correlated tests were run, a few of the significant results are expected by chance, and the calm regime in the COVID window has too few days to say anything.

## 6. Pooled stocks (panel)

In [ ]:
pn = tab[(tab.entity_set == "PANEL") & (tab.regime_type == "volatility") & (tab.horizon == 1)]
display(pn.pivot_table(index=["window", "model"], columns="regime", values="oos_r2")[ra.VOL_ORDER].round(3))

**What we see.** For the pooled stocks the gains over the zero forecast are smaller than for the market proxy, but the machine-learning models are still positive in every regime.

**What it means.** Averaging across stocks makes the market proxy easier to forecast than a single stock, which is the usual pattern.

## 7. Summary and caveats
- Gains over the zero forecast are small in absolute terms, and the machine-learning models beat the simple AR model in most regimes.
- In the post-gap window predictability is highest in calm markets and lowest in turbulent ones. In the COVID window the calm group is too small to judge, but turbulent days are again at or near the weakest for the machine-learning models at 1 day ahead.
- The simple persistence forecast is a poor benchmark and is much worse than predicting zero outside calm periods.
- These numbers come from a local run of `python run.py models` with the models registered at the time. `template_ar5` is a placeholder; the real AR, ARMA and GJR-GARCH models are not registered yet. Re-run this notebook (and `python -m src.eda.regime_analysis`) when they are added, and again when Teddy commits the final predictions.
- Nothing here chooses or tunes a model; the test windows are only used to describe errors.